# LumenY 7 — Binary Directional Classifier (15M) — 7 Majors

**Approach:** Binary classification on 15M bars.

**Label:**
- `+1` if return_15M > spread (clear up move)
- `-1` if return_15M < -spread (clear down move)
- `0` ambiguous (|return| <= spread, no trade)

**Model:** Two LightGBM binary classifiers — one for UP, one for DOWN.

**Signal logic:** Trade when P(up) or P(down) exceeds threshold.

**Features:** features_7 microstructure (64 features, 5M bars, filtered to 15M-aligned)

**Models saved to:** `backend/models_7/classifier/`

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import roc_auc_score, classification_report

FEATURES_DIR = Path('../backend/data/features_7')
MODELS_DIR   = Path('../backend/models_7/classifier')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END  = '2024-06-30'
SPREAD     = 0.00010   # label threshold — 1 pip on majors

print('Ready.')
print(f'Training cutoff: {TRAIN_END}')
print(f'Spread threshold: {SPREAD}')

## 1. Load Data & Build Labels

In [ ]:
df_all = pd.read_parquet(FEATURES_DIR / 'all_pairs_microstructure.parquet')

# Filter to 15M-aligned bars only
df_all = df_all[df_all.index.minute.isin([0, 15, 30, 45])].copy()
print(f'15M-aligned rows: {len(df_all):,}')

# Build directional label
ret = df_all['label_15m']
df_all['label'] = 0
df_all.loc[ret >  SPREAD, 'label'] = 1   # UP
df_all.loc[ret < -SPREAD, 'label'] = -1  # DOWN

# Feature columns
drop_cols    = ['label_5m', 'label_15m', 'label_1h', 'pair', 'label']
feature_cols = [c for c in df_all.columns if c not in drop_cols]

# Train / test split
df_train = df_all[df_all.index <= TRAIN_END].copy()
df_test  = df_all[df_all.index >  TRAIN_END].copy()

# Class balance
for name, df in [('Train', df_train), ('Test', df_test)]:
    vc = df['label'].value_counts().sort_index()
    total = len(df)
    print(f'{name}: UP={vc.get(1,0):,} ({100*vc.get(1,0)/total:.1f}%)  '
          f'DOWN={vc.get(-1,0):,} ({100*vc.get(-1,0)/total:.1f}%)  '
          f'SKIP={vc.get(0,0):,} ({100*vc.get(0,0)/total:.1f}%)  '
          f'Total={total:,}')

print(f'\nFeatures: {len(feature_cols)}')
print(f'Date range: {df_all.index.min().date()} to {df_all.index.max().date()}')

## 2. Walk-Forward Splits

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

# Use all rows for feature matrix (label=0 rows included as negatives for both classifiers)
valid_mask = df_train['label_15m'].notna()
X_clean    = df_train[feature_cols][valid_mask].ffill().fillna(0)
y_clean    = df_train['label'][valid_mask]

splits = walk_forward_splits(len(X_clean))
print(f'Training rows: {len(X_clean):,}')
print(f'Walk-forward splits: {len(splits)}')
for i, (tr_idx, te_idx) in enumerate(splits):
    print(f'  Fold {i+1}: train -> {X_clean.index[tr_idx[-1]].date()} '
          f'({len(tr_idx):,}) | test {X_clean.index[te_idx[0]].date()} '
          f'-> {X_clean.index[te_idx[-1]].date()} ({len(te_idx):,})')

## 3. Model Config

Two binary classifiers:
- **UP model:** P(return > spread) — label_up = 1 if label==+1, else 0
- **DOWN model:** P(return < -spread) — label_down = 1 if label==-1, else 0

Both treat label=0 bars as negatives (ambiguous bars are non-events).

In [ ]:
def get_clf_params():
    return {
        'objective':         'binary',
        'metric':            'auc',
        'boosting_type':     'gbdt',
        'n_estimators':      3000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'is_unbalance':      True,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
        'device':            'gpu',
    }

print('Params ready.')

## 4. Train UP & DOWN Classifiers

In [ ]:
# Binary targets
y_up   = (y_clean ==  1).astype(int)  # 1 = up move, 0 = everything else
y_down = (y_clean == -1).astype(int)  # 1 = down move, 0 = everything else

print(f'UP   positives: {y_up.sum():,} ({100*y_up.mean():.1f}%)')
print(f'DOWN positives: {y_down.sum():,} ({100*y_down.mean():.1f}%)')

results_cv = {}

for direction, y_bin in [('UP', y_up), ('DOWN', y_down)]:
    print(f'\n--- Training {direction} classifier ---')
    params     = get_clf_params()
    oof_proba  = np.full(len(X_clean), np.nan)
    best_iters = []
    aucs       = []

    for fold, (tr_idx, te_idx) in enumerate(splits):
        X_tr, y_tr = X_clean.iloc[tr_idx], y_bin.iloc[tr_idx]
        X_te, y_te = X_clean.iloc[te_idx], y_bin.iloc[te_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])

        proba = model.predict_proba(X_te)[:, 1]
        oof_proba[te_idx] = proba
        auc = roc_auc_score(y_te, proba)
        aucs.append(auc)
        best_iters.append(model.best_iteration_)
        print(f'  Fold {fold+1}: AUC={auc:.4f}, iters={model.best_iteration_}')

    # Filter out degenerate folds (< 10 iters)
    valid_iters = [x for x in best_iters if x >= 10]
    avg_iter    = max(100, int(np.mean(valid_iters))) if valid_iters else 100
    print(f'  CV AUC: {np.mean(aucs):.4f} | Valid iters: {valid_iters} -> avg={avg_iter}')

    # Train final model on all training data
    final = lgb.LGBMClassifier(**{**params, 'n_estimators': avg_iter})
    final.fit(X_clean, y_bin)

    joblib.dump({
        'model':        final,
        'direction':    direction,
        'feature_cols': feature_cols,
        'train_end':    TRAIN_END,
        'spread':       SPREAD,
        'cv_auc':       np.mean(aucs),
        'n_iters':      avg_iter,
    }, MODELS_DIR / f'model_15M_{direction}.joblib')

    results_cv[direction] = {'aucs': aucs, 'iters': best_iters, 'oof': oof_proba}
    print(f'  Saved model_15M_{direction}.joblib')
    del final, model; gc.collect()

print('\nTraining complete.')

## 5. Test Set Evaluation

In [ ]:
# Prepare test set
valid_test   = df_test['label_15m'].notna()
X_test_clean = df_test[feature_cols][valid_test].ffill().fillna(0)
y_test_label = df_test['label'][valid_test]
y_test_ret   = df_test['label_15m'][valid_test]
pairs_test   = df_test['pair'][valid_test]

n_test_days  = (X_test_clean.index.max() - X_test_clean.index.min()).days

# Load models and predict
proba_up   = joblib.load(MODELS_DIR / 'model_15M_UP.joblib')['model'].predict_proba(X_test_clean)[:, 1]
proba_down = joblib.load(MODELS_DIR / 'model_15M_DOWN.joblib')['model'].predict_proba(X_test_clean)[:, 1]

results = pd.DataFrame({
    'ret':        y_test_ret.values,
    'label':      y_test_label.values,
    'proba_up':   proba_up,
    'proba_down': proba_down,
    'pair':       pairs_test.values,
}, index=X_test_clean.index)

# Signal: take the stronger of the two probabilities
results['signal']    = 0
results['signal_p']  = np.maximum(results['proba_up'], results['proba_down'])
results.loc[results['proba_up']   > results['proba_down'], 'signal'] =  1
results.loc[results['proba_down'] > results['proba_up'],   'signal'] = -1

print(f'Test set: {len(results):,} rows ({results.index.min().date()} -> {results.index.max().date()})')
print(f'Baseline (no model): UP={100*(results["label"]==1).mean():.1f}%  '
      f'DOWN={100*(results["label"]==-1).mean():.1f}%  '
      f'SKIP={100*(results["label"]==0).mean():.1f}%')

# Probability threshold sweep
print(f'\n{"Threshold":>10} {"Trades":>8} {"Tr/day":>8} {"WinRate":>9} '
      f'{"EV/trade":>11} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 72)

for thresh in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    mask = results['signal_p'] > thresh
    if mask.sum() < 10:
        continue
    s   = results[mask]
    # P&L: signal direction * actual return - spread
    pnl = s['signal'] * s['ret'] - SPREAD
    wr  = (s['signal'] == s['label']).mean()
    ev  = pnl.mean()
    tot = pnl.sum()
    shr = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24 * 4) if pnl.std() > 0 else 0
    tpd = mask.sum() / n_test_days
    flag = ' <<<' if ev > 0 else ''
    print(f'{thresh:>10.2f} {mask.sum():>8,} {tpd:>8.1f} {wr:>8.1%} '
          f'{ev:>11.6f} {tot:>12.4f} {shr:>8.2f}{flag}')

print(f'\nSpread cost: {SPREAD}')

## 6. Per-Pair Breakdown

In [ ]:
BEST_THRESH = 0.55  # adjust after seeing sweep results above

filtered = results[results['signal_p'] > BEST_THRESH]
print(f'Per-pair results (signal_p > {BEST_THRESH}):')
print(f'\n{"Pair":<10} {"Trades":>8} {"Tr/day":>8} {"WinRate":>9} '
      f'{"EV/trade":>11} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 72)

for pair in sorted(filtered['pair'].unique()):
    p   = filtered[filtered['pair'] == pair]
    pnl = p['signal'] * p['ret'] - SPREAD
    wr  = (p['signal'] == p['label']).mean()
    shr = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24 * 4) if pnl.std() > 0 else 0
    tpd = len(p) / n_test_days
    flag = ' <<<' if pnl.mean() > 0 else ''
    print(f'{pair:<10} {len(p):>8,} {tpd:>8.1f} {wr:>8.1%} '
          f'{pnl.mean():>11.6f} {pnl.sum():>12.4f} {shr:>8.2f}{flag}')

## 7. Edge by Hour of Day

In [ ]:
filtered = results[results['signal_p'] > BEST_THRESH].copy()
filtered['hour'] = filtered.index.hour
filtered['pnl']  = filtered['signal'] * filtered['ret'] - SPREAD
filtered['correct'] = (filtered['signal'] == filtered['label']).astype(int)

hourly = filtered.groupby('hour').agg(
    trades=('pnl', 'count'),
    wr=('correct', 'mean'),
    ev=('pnl', 'mean'),
    total_pnl=('pnl', 'sum')
).reset_index()

print(f'{"Hour":>6} {"Trades":>8} {"WinRate":>9} {"EV/trade":>11} {"Total PnL":>12}')
print('-' * 52)
for _, row in hourly.iterrows():
    flag = ' <<<' if row['ev'] > 0 else ''
    print(f'{int(row["hour"]):>6} {int(row["trades"]):>8} {row["wr"]:>8.1%} '
          f'{row["ev"]:>11.6f} {row["total_pnl"]:>12.4f}{flag}')

## 8. Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#080c14')

thresholds_plot = [0.45, 0.50, 0.55, 0.60]

for ax, thresh in zip(axes.flatten(), thresholds_plot):
    ax.set_facecolor('#080c14')
    subset  = results[results['signal_p'] > thresh]
    pnl     = subset['signal'] * subset['ret'] - SPREAD
    cum_pnl = pnl.cumsum()

    for pair in sorted(subset['pair'].unique()):
        p        = subset[subset['pair'] == pair]
        pair_pnl = (p['signal'] * p['ret'] - SPREAD).cumsum()
        ax.plot(pair_pnl.index, pair_pnl.values, alpha=0.3, linewidth=0.7)

    ax.plot(cum_pnl.index, cum_pnl.values, color='#4fc3f7', linewidth=2)
    ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')

    wr  = (subset['signal'] == subset['label']).mean()
    ev  = pnl.mean()
    shr = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24 * 4) if pnl.std() > 0 else 0
    ax.set_title(f'Threshold={thresh} ({len(subset):,} trades)\n'
                 f'WR: {wr:.1%}, EV: {ev:.6f}, Sharpe: {shr:.2f}',
                 color='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.set_ylabel('Cum PnL (after spread)', color='white', fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Binary Classifier 15M — Equity Curves on Unseen Test Set',
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.patch.set_facecolor('#080c14')

for ax, direction in zip(axes, ['UP', 'DOWN']):
    bundle     = joblib.load(MODELS_DIR / f'model_15M_{direction}.joblib')
    importance = pd.Series(bundle['model'].feature_importances_, index=feature_cols)
    importance = importance.sort_values(ascending=True).tail(25)

    ax.barh(importance.index, importance.values, color='#4fc3f7', alpha=0.8)
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=7)
    ax.set_title(f'{direction} Classifier — Top 25 Features', color='white', fontsize=11)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Feature Importance — 15M Binary Classifier', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=' * 70)
print('BINARY CLASSIFIER 15M — COMPLETE')
print('=' * 70)

for direction in ['UP', 'DOWN']:
    b = joblib.load(MODELS_DIR / f'model_15M_{direction}.joblib')
    print(f'\n{direction} model: AUC={b["cv_auc"]:.4f}, iters={b["n_iters"]}  '
          f'| {(MODELS_DIR / f"model_15M_{direction}.joblib").stat().st_size/1e6:.1f} MB')

print(f'\n-- Test Set Results --')
for thresh in [0.45, 0.50, 0.55, 0.60, 0.65]:
    mask = results['signal_p'] > thresh
    if mask.sum() < 10: continue
    s   = results[mask]
    pnl = s['signal'] * s['ret'] - SPREAD
    wr  = (s['signal'] == s['label']).mean()
    ev  = pnl.mean()
    shr = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24 * 4) if pnl.std() > 0 else 0
    tpd = mask.sum() / n_test_days
    flag = ' <<<' if ev > 0 else ''
    print(f'  P>{thresh:.2f}: WR={wr:.1%}  EV={ev:.6f}  Sharpe={shr:.2f}  Tr/day={tpd:.1f}{flag}')

print(f'\nTest period: {results.index.min().date()} -> {results.index.max().date()}')
print(f'Training cutoff: {TRAIN_END}')
print(f'Spread threshold: {SPREAD}')
print(f'Features: {len(feature_cols)}')
print(f'Pairs: {sorted(results["pair"].unique())}')